In [41]:
#匯入套件

import pandas as pd
import numpy as np
import warnings
import random
import os

from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

random.seed(42)
warnings.filterwarnings('ignore')

In [42]:
# Mutual Information of Features 

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# 1. 讀取資料
df = pd.read_csv('../data/boy or girl 2025 train_missingValue.csv')
df['yt'] = pd.to_numeric(df['yt'], errors='coerce')

# 2. 過濾不合理的極端值 (將物理不可能的值轉為 NaN)
df['height'] = df['height'].apply(lambda x: x if 100 <= x <= 250 else np.nan)
df['weight'] = df['weight'].apply(lambda x: x if 30 <= x <= 200 else np.nan)
df['iq'] = df['iq'].apply(lambda x: x if 50 <= x <= 200 else np.nan)
df['sleepiness'] = df['sleepiness'].apply(lambda x: x if 0 <= x <= 24 else np.nan)
df['fb_friends'] = df['fb_friends'].apply(lambda x: x if x >= 0 else np.nan)
df['yt'] = df['yt'].apply(lambda x: x if x >= 0 else np.nan)

# 3. 為了計算 MI，必須先暫時補值 (MI 函數無法處理 NaN)
# 這裡只是為了算重要性，所以用最簡單的中位數和眾數來補
df_imputed = df.copy()
for col in ['height', 'weight', 'iq', 'sleepiness', 'fb_friends', 'yt']:
    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

for col in ['star_sign', 'phone_os', 'self_intro']:
    df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mode()[0])
    # 類別型特徵必須轉成數字標籤，MI 函數才能吃
    df_imputed[col] = LabelEncoder().fit_transform(df_imputed[col].astype(str))

# 4. 分離特徵 (X) 與目標 (y)
X = df_imputed.drop(columns=['id', 'gender'])
y = df_imputed['gender']

# 5. 核心步驟：計算 Mutual Information
# random_state=42 確保每次跑出來的結果一致
mi_scores = mutual_info_classif(X, y, random_state=42)

# 將結果整理成好閱讀的 Pandas Series 並排序
mi_scores_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

print("=== 特徵 Information Gain (Mutual Information) ===")
print(mi_scores_series)

=== 特徵 Information Gain (Mutual Information) ===
height        0.241552
weight        0.135601
self_intro    0.046648
fb_friends    0.026675
iq            0.021685
yt            0.017019
star_sign     0.000000
sleepiness    0.000000
phone_os      0.000000
dtype: float64


In [50]:
# Cell 2: 完整資料前處理 (包含物理邊界過濾、10維轉5維 PCA、強化平滑化 Target Encoding)

# 1. 讀取原始資料
train_df = pd.read_csv('../data/boy or girl 2025 train_missingValue.csv')
test_df = pd.read_csv('../data/boy or girl 2025 test no ans_missingValue.csv')

# 2. 清理 phone_os 噪音值 (優化點 4)
def clean_os(val):
    val = str(val).lower()
    if 'apple' in val or 'ios' in val: return 'Apple'
    if 'android' in val: return 'Android'
    return 'Other'

train_df['phone_os'] = train_df['phone_os'].apply(clean_os)
test_df['phone_os'] = test_df['phone_os'].apply(clean_os)

# 3. 物理邊界強化 (修正：fb_friends 限制在 5000 以內，YT 加上防溢位處理)
def apply_advanced_bounds(df):
    df_clean = df.copy()
    # 確保數值型轉型成功
    df_clean['yt'] = pd.to_numeric(df_clean['yt'], errors='coerce')
    
    # 嚴格過濾物理不可能的極端值 (優化點 1)
    df_clean['height'] = df_clean['height'].apply(lambda x: x if 100 <= x <= 250 else np.nan)
    df_clean['weight'] = df_clean['weight'].apply(lambda x: x if 30 <= x <= 200 else np.nan)
    df_clean['iq'] = df_clean['iq'].apply(lambda x: x if 50 <= x <= 200 else np.nan)
    df_clean['sleepiness'] = df_clean['sleepiness'].apply(lambda x: x if 0 <= x <= 24 else np.nan)
    
    # Facebook 好友上限約 5000，且過濾掉負值
    df_clean['fb_friends'] = df_clean['fb_friends'].apply(lambda x: x if 0 <= x <= 5000 else np.nan)
    
    # 針對 yt 加入 1e15 限制以防後續 float32 溢位
    df_clean['yt'] = df_clean['yt'].apply(lambda x: x if 0 <= x <= 1e15 else np.nan)
    return df_clean

train_df = apply_advanced_bounds(train_df)
test_df = apply_advanced_bounds(test_df)

# 4. self_intro Embedding 與 PCA 降維 (優化點 5：降至 5 維以防 Overfitting)
print("載入 SentenceTransformer 並執行 5 維 PCA 降維中...")
train_df['self_intro'] = train_df['self_intro'].fillna('')
test_df['self_intro'] = test_df['self_intro'].fillna('')

embedder = SentenceTransformer('all-MiniLM-L6-v2')
train_emb = embedder.encode(train_df['self_intro'].tolist())
test_emb = embedder.encode(test_df['self_intro'].tolist())

# 從 10 降回 5 維，減少小樣本下的維度災難
pca = PCA(n_components=5, random_state=42)
train_pca = pca.fit_transform(train_emb)
test_pca = pca.transform(test_emb)

for i in range(5):
    train_df[f'intro_pca_{i}'] = train_pca[:, i]
    test_df[f'intro_pca_{i}'] = test_pca[:, i]

# 5. 保留類別特徵並進行 Target Encoding (優化點 2：增加 Smoothing 到 20)
# 分離 X 與 y
X = train_df.drop(columns=['id', 'gender', 'self_intro'])
y = train_df['gender']
X_test_submission = test_df.drop(columns=['id', 'gender', 'self_intro'])
test_ids = test_df['id']

# 轉換 Target 變數 (0=男, 1=女)
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

# 執行 Target Encoding，將平滑係數提高至 20 以對抗樣本不足的問題
print("執行 Target Encoding (Smoothing=20)...")
te = TargetEncoder(cols=['star_sign', 'phone_os'], smoothing=20)
X = te.fit_transform(X, y_encoded)
X_test_submission = te.transform(X_test_submission)

# 6. 建立缺失值指示器 (保留基礎物理特徵的缺失紀錄)
for col in ['weight', 'height', 'yt']:
    X[f'{col}_is_missing'] = X[col].isnull().astype(int)
    X_test_submission[f'{col}_is_missing'] = X_test_submission[col].isnull().astype(int)

print(f"✅ Cell 2 處理完成！目前特徵總數: {X.shape[1]}")

載入 SentenceTransformer 並執行 5 維 PCA 降維中...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10871.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


執行 Target Encoding (Smoothing=20)...
✅ Cell 2 處理完成！目前特徵總數: 16


In [51]:
print("開始使用 Random Forest 演算法插補缺失值...")

rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
    random_state=42, 
    max_iter=10
)

# 注意：Imputer 產出的是 Numpy Array，我們將它轉回 DataFrame 以方便後續計算 BMI
X_imputed_array = rf_imputer.fit_transform(X)
X_test_imputed_array = rf_imputer.transform(X_test_submission)

# 轉回 Pandas DataFrame
X_imputed = pd.DataFrame(X_imputed_array, columns=X.columns)
X_test_imputed = pd.DataFrame(X_test_imputed_array, columns=X_test_submission.columns)

print("✅ 缺失值插補完成！")

開始使用 Random Forest 演算法插補缺失值...
✅ 缺失值插補完成！


In [52]:
print("開始建立進階衍生特徵 (BMI, Log Transform)...")

def create_advanced_features(df):
    df_new = df.copy()
    
    # 1. 建立 BMI 特徵 (防呆：確保身高不為 0，避免除以零錯誤)
    df_new['height'] = df_new['height'].clip(lower=1) # 身高至少大於 1cm
    df_new['BMI'] = df_new['weight'] / ((df_new['height'] / 100) ** 2)
    
    # 2. 對極端右偏的數值進行 Log1p 轉換 (Log(1+x))
    # clip(lower=0) 確保沒有負數，因為 log 裡面不能是負數
    df_new['yt_log'] = np.log1p(df_new['yt'].clip(lower=0))
    df_new['fb_friends_log'] = np.log1p(df_new['fb_friends'].clip(lower=0))
    
    # 捨棄原本未經 log 轉換的欄位，減少特徵共線性
    df_new = df_new.drop(columns=['yt', 'fb_friends'])
    
    return df_new

X_imputed_eng = create_advanced_features(X_imputed)
X_test_imputed_eng = create_advanced_features(X_test_imputed)

print(f"✅ 特徵工程完成！目前的特徵數量為: {X_imputed_eng.shape[1]}")

開始建立進階衍生特徵 (BMI, Log Transform)...
✅ 特徵工程完成！目前的特徵數量為: 17


In [53]:
print("開始剔除無效特徵...")

# 修改：把 BMI, yt_log, fb_friends_log 從這裡移除，把它們保留給模型
useless_features = [
    'weight_is_missing' # 原本被你點名的其他無效特徵可以放在這
]

X_pruned = X_imputed_eng.drop(columns=useless_features, errors='ignore')
X_test_pruned = X_test_imputed_eng.drop(columns=useless_features, errors='ignore')

print(f"✅ 剔除完成！原本有 {X_imputed_eng.shape[1]} 個特徵，現在剩餘 {X_pruned.shape[1]} 個特徵。")

開始剔除無效特徵...
✅ 剔除完成！原本有 17 個特徵，現在剩餘 16 個特徵。


In [54]:
# 修改：徹底移除 MinMaxScaler，直接用 X_pruned 做資料切割

X_train, X_val, y_train, y_val = train_test_split(
    X_pruned, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
)
print("✅ 資料切割完成！")

✅ 資料切割完成！


In [ ]:
#train version 1- 3 kind of tree-based

from sklearn.model_selection import cross_validate
from sklearn.metrics import classification_report

print("6. 訓練 Ensemble 模型 (開啟 XGBoost GPU 加速)...")
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, max_depth=4, learning_rate=0.05, n_estimators=150, tree_method='hist', device='cuda')
lgbm_clf = LGBMClassifier(random_state=42, max_depth=4, learning_rate=0.05, n_estimators=150, verbose=-1)
rf_clf = RandomForestClassifier(random_state=42, max_depth=6, n_estimators=150)

ensemble_model = VotingClassifier(
    estimators=[('XGBoost', xgb_clf), ('LightGBM', lgbm_clf), ('RandomForest', rf_clf)], 
    voting='soft'
)

# 定義我們要觀察的多個指標
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'f1': 'f1',           # 如果是二元分類，預設看類別 1 的 F1
    'log_loss': 'neg_log_loss' # sklearn 習慣用負的 Log Loss，我們等一下轉正
}

print("進行 Stratified 10-Fold 交叉驗證中 (多指標評估)...")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 使用 cross_validate 一次算出所有指標
cv_results = cross_validate(ensemble_model, X_train, y_train, cv=skf, scoring=scoring_metrics)

# 印出漂亮的報表
print("\n📊 === 10-Fold CV 綜合評估報告 ===")
print(f"Accuracy (準確率) : {cv_results['test_accuracy'].mean():.4f} (std: {cv_results['test_accuracy'].std():.4f})")
print(f"ROC-AUC  (排序力) : {cv_results['test_roc_auc'].mean():.4f} (std: {cv_results['test_roc_auc'].std():.4f})  <-- 越高越好")
print(f"F1-Score (平衡度) : {cv_results['test_f1'].mean():.4f} (std: {cv_results['test_f1'].std():.4f})")
# 把負號拿掉，讓它變回正常的 Log Loss
print(f"Log Loss (誤差值) : {-cv_results['test_log_loss'].mean():.4f} (std: {cv_results['test_log_loss'].std():.4f})  <-- 越低越好")

# 驗證集預測與詳細報告
print("\n🎯 === 15% 驗證集 (Validation Set) 詳細報告 ===")
ensemble_model.fit(X_train, y_train)
y_val_pred = ensemble_model.predict(X_val)
y_val_proba = ensemble_model.predict_proba(X_val)[:, 1] # 取得預測為 1 的機率

print(classification_report(y_val, y_val_pred))

6. 訓練 Ensemble 模型 (開啟 XGBoost GPU 加速)...
進行 Stratified 10-Fold 交叉驗證中 (多指標評估)...

📊 === 10-Fold CV 綜合評估報告 ===
Accuracy (準確率) : 0.8856 (std: 0.0553)
ROC-AUC  (排序力) : 0.9164 (std: 0.0678)  <-- 越高越好
F1-Score (平衡度) : 0.7641 (std: 0.1121)
Log Loss (誤差值) : 0.3077 (std: 0.1122)  <-- 越低越好

🎯 === 15% 驗證集 (Validation Set) 詳細報告 ===
              precision    recall  f1-score   support

           0       0.96      0.90      0.92        48
           1       0.74      0.88      0.80        16

    accuracy                           0.89        64
   macro avg       0.85      0.89      0.86        64
weighted avg       0.90      0.89      0.89        64



In [39]:
#train version 2 - 加入 MLP、Logistic Regression

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

print("6. 訓練異質 Ensemble 模型 (XGB + MLP + LR)...")

# 1. XGBoost (樹狀模型，不需縮放，維持原樣)
xgb_clf = XGBClassifier(
    use_label_encoder=False, eval_metric='logloss', random_state=42, 
    max_depth=4, learning_rate=0.05, n_estimators=150, 
    tree_method='hist', device='cuda'
)

# 2. MLP (神經網路，極度依賴縮放 -> 使用 Pipeline 包裝 StandardScaler)
# hidden_layer_sizes=(32, 16) 代表兩層隱藏層，適合這種小型特徵數量
mlp_clf = make_pipeline(
    StandardScaler(), 
    MLPClassifier(random_state=42, hidden_layer_sizes=(32, 16), max_iter=500, early_stopping=True)
)

# 3. Logistic Regression (線性模型，極度依賴縮放 -> 使用 Pipeline 包裝 StandardScaler)
lr_clf = make_pipeline(
    StandardScaler(), 
    LogisticRegression(random_state=42, max_iter=500, class_weight='balanced')
)

# 4. 組合成為軟投票分類器
ensemble_model = VotingClassifier(
    estimators=[
        ('XGBoost', xgb_clf), 
        ('MLP', mlp_clf), 
        ('LogReg', lr_clf)
    ], 
    voting='soft'
)
# 定義我們要觀察的多個指標
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'f1': 'f1',           # 如果是二元分類，預設看類別 1 的 F1
    'log_loss': 'neg_log_loss' # sklearn 習慣用負的 Log Loss，我們等一下轉正
}

print("進行 Stratified 10-Fold 交叉驗證中 (多指標評估)...")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 使用 cross_validate 一次算出所有指標
cv_results = cross_validate(ensemble_model, X_train, y_train, cv=skf, scoring=scoring_metrics)

# 印出漂亮的報表
print("\n📊 === 10-Fold CV 綜合評估報告 ===")
print(f"Accuracy (準確率) : {cv_results['test_accuracy'].mean():.4f} (std: {cv_results['test_accuracy'].std():.4f})")
print(f"ROC-AUC  (排序力) : {cv_results['test_roc_auc'].mean():.4f} (std: {cv_results['test_roc_auc'].std():.4f})  <-- 越高越好")
print(f"F1-Score (平衡度) : {cv_results['test_f1'].mean():.4f} (std: {cv_results['test_f1'].std():.4f})")
# 把負號拿掉，讓它變回正常的 Log Loss
print(f"Log Loss (誤差值) : {-cv_results['test_log_loss'].mean():.4f} (std: {cv_results['test_log_loss'].std():.4f})  <-- 越低越好")

# 驗證集預測與詳細報告
print("\n🎯 === 15% 驗證集 (Validation Set) 詳細報告 ===")
ensemble_model.fit(X_train, y_train)
y_val_pred = ensemble_model.predict(X_val)
y_val_proba = ensemble_model.predict_proba(X_val)[:, 1] # 取得預測為 1 的機率

print(classification_report(y_val, y_val_pred))

6. 訓練異質 Ensemble 模型 (XGB + MLP + LR)...
進行 Stratified 10-Fold 交叉驗證中 (多指標評估)...

📊 === 10-Fold CV 綜合評估報告 ===
Accuracy (準確率) : 0.8831 (std: 0.0592)
ROC-AUC  (排序力) : 0.9014 (std: 0.0826)  <-- 越高越好
F1-Score (平衡度) : 0.7678 (std: 0.1169)
Log Loss (誤差值) : 0.3645 (std: 0.0531)  <-- 越低越好

🎯 === 15% 驗證集 (Validation Set) 詳細報告 ===
              precision    recall  f1-score   support

           0       0.94      0.92      0.93        48
           1       0.76      0.81      0.79        16

    accuracy                           0.89        64
   macro avg       0.85      0.86      0.86        64
weighted avg       0.89      0.89      0.89        64



In [55]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import classification_report
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

print("6. 訓練 Stacking 模型 (處理 Class Imbalance)...")

# 計算男女比例常數 (性別1/性別2) 用於 XGBoost
ratio = (y_encoded == 0).sum() / (y_encoded == 1).sum()

# Base Learners
# 優化點 3: 加入權重處理
xgb_clf = XGBClassifier(
    scale_pos_weight=ratio, 
    eval_metric='logloss', 
    random_state=42, 
    max_depth=4, 
    tree_method='hist', 
    device='cuda'
)

lgbm_clf = LGBMClassifier(
    class_weight='balanced', 
    random_state=42, 
    max_depth=4, 
    verbose=-1
)

rf_clf = RandomForestClassifier(
    class_weight='balanced', 
    random_state=42, 
    max_depth=6
)

# Meta Learner
meta_learner = LogisticRegression(class_weight='balanced', random_state=42)

stacking_model = StackingClassifier(
    estimators=[('XGB', xgb_clf), ('LGBM', lgbm_clf), ('RF', rf_clf)],
    final_estimator=meta_learner,
    cv=5
)

# 評估
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = cross_validate(stacking_model, X_train, y_train, cv=skf, scoring=scoring_metrics)


print("\n📊 === Stacking 10-Fold CV 綜合評估報告 ===")
print(f"Accuracy (準確率) : {cv_results['test_accuracy'].mean():.4f} (std: {cv_results['test_accuracy'].std():.4f})")
print(f"ROC-AUC  (排序力) : {cv_results['test_roc_auc'].mean():.4f} (std: {cv_results['test_roc_auc'].std():.4f})")
print(f"F1-Score (平衡度) : {cv_results['test_f1'].mean():.4f} (std: {cv_results['test_f1'].std():.4f})")
print(f"Log Loss (誤差值) : {-cv_results['test_log_loss'].mean():.4f} (std: {cv_results['test_log_loss'].std():.4f})")

# 記得後面的 Cell 8 預測存檔也要把模型名稱改成 stacking_model 哦！
# 例如：test_predictions_encoded = stacking_model.predict(X_test_final)

6. 訓練 Stacking 模型 (處理 Class Imbalance)...

📊 === Stacking 10-Fold CV 綜合評估報告 ===
Accuracy (準確率) : 0.8775 (std: 0.0394)
ROC-AUC  (排序力) : 0.9208 (std: 0.0623)
F1-Score (平衡度) : 0.7671 (std: 0.0797)
Log Loss (誤差值) : 0.3676 (std: 0.0778)


In [59]:
# ---------------------------------------------------------
# 最後一個 Cell: 預測與存檔 (修正版)
# ---------------------------------------------------------
import os
import pandas as pd

print(f"目前訓練集特徵數: {X_pruned.shape[1]}") 
print(f"目前測試集特徵數: {X_test_pruned.shape[1]}") 

# 🚨 關鍵步驟：用所有資料正式訓練模型！
print("正式訓練 Stacking 模型中 (這會花一點時間)...")
stacking_model.fit(X_pruned, y_encoded)
print("訓練完成！")

# 1. 預測時餵給模型 X_test_pruned
print("進行測試集預測中...")
test_predictions_encoded = stacking_model.predict(X_test_pruned) 

# 2. 將 0, 1 轉回原始的性別標籤
test_predictions_original = le_y.inverse_transform(test_predictions_encoded)

# 3. 建立 Submission DataFrame
submission_df = pd.DataFrame({
    'id': test_ids, 
    'gender': test_predictions_original
})

# 4. 自動建立資料夾並存檔
output_dir = '../results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, 'submission_stacking_v2.csv')
submission_df.to_csv(file_path, index=False)

print(f"🏆 預測成功！檔案已儲存至: {file_path}")
print("前五筆預測結果：")
print(submission_df.head())

目前訓練集特徵數: 16
目前測試集特徵數: 16
正式訓練 Stacking 模型中 (這會花一點時間)...
訓練完成！
進行測試集預測中...
🏆 預測成功！檔案已儲存至: ../results\submission_stacking_v2.csv
前五筆預測結果：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       2
